# 01 -- Ingesta y Limpieza de Datos NYC 311

**Objetivo**: Descargar solicitudes de servicio NYC 311 del ano 2024 via la API Socrata (SODA), limpiar valores nulos, estandarizar nombres y deduplicar registros.

**Fuente**: [NYC Open Data - 311 Service Requests](https://data.cityofnewyork.us/Social-Services/311-Service-Requests-from-2010-to-Present/erm2-nwe9) (dataset `erm2-nwe9`)

**Periodo**: Enero - Diciembre 2024

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 30)
pd.set_option('display.float_format', '{:.2f}'.format)

DATA_RAW = Path('../data/raw')
DATA_PROC = Path('../data/processed')
DATA_PROC.mkdir(parents=True, exist_ok=True)

print('Librerias cargadas')

## 1. Descarga via Socrata API

La API SODA permite consultar datos publicos de NYC sin necesidad de token. Paginamos en bloques de 50,000 filas para manejar el volumen (~3.5M solicitudes en 2024).

In [ ]:
# Si ya tenemos el CSV descargado, cargar directamente
raw_path = DATA_RAW / 'nyc311_raw.csv'

if raw_path.exists():
    df_raw = pd.read_csv(raw_path, low_memory=False)
    print(f'Datos cargados desde CSV: {len(df_raw):,} filas')
else:
    print('Archivo no encontrado. Ejecuta data-pipeline/01_download.py primero.')
    print('  python data-pipeline/01_download.py')
    df_raw = pd.DataFrame()

In [ ]:
print(f'Dimensiones: {df_raw.shape}')
print(f'\nColumnas ({len(df_raw.columns)}):')
for col in df_raw.columns:
    print(f'  {col}: {df_raw[col].dtype} -- {df_raw[col].nunique():,} unicos, {df_raw[col].isna().sum():,} nulos ({df_raw[col].isna().mean():.1%})')

In [ ]:
df_raw.head(3)

## 2. Limpieza de Datos

### 2.1 Conversion de fechas

In [ ]:
df = df_raw.copy()

# Parsear fechas
for col in ['created_date', 'closed_date', 'due_date']:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')
        print(f'{col}: {df[col].notna().sum():,} validos, {df[col].isna().sum():,} nulos')

# Eliminar filas sin fecha de creacion
antes = len(df)
df = df.dropna(subset=['created_date'])
print(f'\nFilas eliminadas por created_date nulo: {antes - len(df):,}')

### 2.2 Estandarizacion de campos categoricos

In [ ]:
# Borough: strip, title case, mapear Unspecified
if 'borough' in df.columns:
    df['borough'] = df['borough'].str.strip().str.title()
    df.loc[df['borough'].isin(['Unspecified', '0', '']), 'borough'] = np.nan
    print('Municipios:', df['borough'].value_counts().to_dict())

# Agency: strip, uppercase
if 'agency' in df.columns:
    df['agency'] = df['agency'].str.strip().str.upper()

# Agency name: strip, title
if 'agency_name' in df.columns:
    df['agency_name'] = df['agency_name'].str.strip()

# Status: strip, title
if 'status' in df.columns:
    df['status'] = df['status'].str.strip().str.title()
    print('\nEstatus:', df['status'].value_counts().to_dict())

### 2.3 Anomalias en fechas de cierre

In [ ]:
# Si closed_date < created_date, es una anomalia
if 'closed_date' in df.columns:
    anomalias = df['closed_date'] < df['created_date']
    n_anomalias = anomalias.sum()
    print(f'Anomalias (closed < created): {n_anomalias:,} ({n_anomalias/len(df):.2%})')
    df.loc[anomalias, 'closed_date'] = np.nan

### 2.4 Deduplicacion

In [ ]:
antes = len(df)
df = df.drop_duplicates(subset=['unique_key'], keep='first')
print(f'Duplicados eliminados: {antes - len(df):,}')
print(f'Filas finales: {len(df):,}')

## 3. Resumen de Calidad de Datos

In [ ]:
# Resumen de nulos por columna
null_summary = pd.DataFrame({
    'nulos': df.isna().sum(),
    'pct_nulo': (df.isna().mean() * 100).round(1),
    'tipo': df.dtypes
}).sort_values('pct_nulo', ascending=False)

null_summary[null_summary['nulos'] > 0]

In [ ]:
# Visualizar completitud
fig = px.bar(
    null_summary.reset_index(),
    x='pct_nulo', y='index',
    orientation='h',
    title='Porcentaje de Valores Nulos por Columna',
    labels={'pct_nulo': '% Nulo', 'index': 'Columna'},
    color='pct_nulo',
    color_continuous_scale='RdYlGn_r'
)
fig.update_layout(height=400, template='plotly_dark')
fig.show()

## 4. Exportar Datos Limpios

In [ ]:
output_path = DATA_PROC / 'nyc311_clean.parquet'
df.to_parquet(output_path, engine='pyarrow', index=False)
print(f'Datos limpios guardados en: {output_path}')
print(f'  Filas: {len(df):,}')
print(f'  Columnas: {len(df.columns)}')
print(f'  Tamano: {output_path.stat().st_size / 1024 / 1024:.1f} MB')

---

**Siguiente**: [02_eda_exploratorio.ipynb](02_eda_exploratorio.ipynb) -- Analisis exploratorio de distribuciones y patrones temporales.